In [ ]:
#
# ⚡ UNIVERSAL FIRST CELL - Run this FIRST!
# Compatible with: Google Colab, GitHub Codespaces, Local
#

import sys, os, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Clone repo
    !git clone https://github.com/Aidas-dev/computer-data-analysis-report.git /content/repo
    %cd /content/repo
    !pip install -q pandas numpy scikit-learn matplotlib seaborn yfinance statsmodels
    # DVC pull
    !pip install -q dvc[s3]
    import os
    # Set OCI creds from colab secrets or env
    !dvc pull data/processed/event_study_dataset.csv.dvc
    !dvc pull data/processed/quarterly_panel_updated.csv.dvc
    !dvc pull data/processed/timeseries_features_updated.csv.dvc
    !dvc pull data/processed/fractracker_gdelt_deduped.csv.dvc
    DATA_DIR = '/content/repo/data'
else:
    DATA_DIR = 'data'

print(f"\u2705 DATA_DIR = {DATA_DIR}")
print(f"\ud83d\ude80 Environment: {'Colab' if IN_COLAB else 'Local'}")

# 18 - Event Study & ML Classification

**Goal**: Run event study (CAR analysis) and ML classification on the 175 deduped FracTracker-GDELT matched events.

**Data sources**:
- `event_study_dataset.csv` — daily stock price data around announcement events
- `fractracker_gdelt_deduped.csv` — 175 matched FracTracker-GDELT events
- `quarterly_panel_updated.csv` — quarterly financial metrics
- `timeseries_features_updated.csv` — daily technical features

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve,
                             confusion_matrix, ConfusionMatrixDisplay,
                             classification_report)
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 300
print("\u2705 Libraries imported")

In [ ]:
# Load event study dataset
es = pd.read_csv(f'{DATA_DIR}/processed/event_study_dataset.csv')
print(f"\ud83d\udcca Event study data: {len(es)} rows \u00d7 {len(es.columns)} cols")

# Load deduped matched events
deduped = pd.read_csv(f'{DATA_DIR}/processed/fractracker_gdelt_deduped.csv')
print(f"\ud83d\udcca Deduped matched events: {len(deduped)} rows \u00d7 {len(deduped.columns)} cols")

# Load quarterly panel
qp = pd.read_csv(f'{DATA_DIR}/processed/quarterly_panel_updated.csv')
print(f"\ud83d\udcca Quarterly panel: {len(qp)} rows \u00d7 {len(qp.columns)} cols")

# Load timeseries features
tsf = pd.read_csv(f'{DATA_DIR}/processed/timeseries_features_updated.csv')
print(f"\ud83d\udcca Timeseries features: {len(tsf)} rows \u00d7 {len(tsf.columns)} cols")

# Parse dates
es['Date'] = pd.to_datetime(es['Date'], errors='coerce')
es['announcement_date'] = pd.to_datetime(es['announcement_date'], errors='coerce')
print("\u2705 Data loaded and dates parsed")

## Exploratory Data Analysis

In [ ]:
# Get unique events (one row per unique announcement)
events = es[['ticker', 'announcement_date', 'ft_status', 'ft_facility_name',
             'bp_mw_capacity', 'gdelt_company', 'bp_tone']].drop_duplicates().copy()
print(f"Total unique events: {len(events)}")
print()

# Event count by status
status_counts = events['ft_status'].value_counts()
print("Event count by status:")
for s, c in status_counts.items():
    print(f"  {s}: {c}")

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336', '#607D8B']
bars = ax.bar(status_counts.index, status_counts.values, color=colors[:len(status_counts)])
ax.set_xlabel('Status')
ax.set_ylabel('Count')
ax.set_title('Event Count by FracTracker Status')
ax.tick_params(axis='x', rotation=45)
for bar, val in zip(bars, status_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(val), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('reports/figures/fig12_events_by_status.png', dpi=300)
plt.show()
print("\u2705 Saved reports/figures/fig12_events_by_status.png")

In [ ]:
# Event count by company
company_counts = events['gdelt_company'].value_counts().head(15)
print("Top 15 companies by event count:")
for c, n in company_counts.items():
    print(f"  {c}: {n}")

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(range(len(company_counts)), company_counts.values, color='#2196F3')
ax.set_xticks(range(len(company_counts)))
ax.set_xticklabels(company_counts.index, rotation=45, ha='right')
ax.set_xlabel('Company')
ax.set_ylabel('Event Count')
ax.set_title('Event Count by Company (Top 15)')
for bar, val in zip(bars, company_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            str(val), ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig('reports/figures/fig12_events_by_company.png', dpi=300)
plt.show()
print("\u2705 Saved reports/figures/fig12_events_by_company.png")

In [ ]:
# MW capacity distribution by status
mw_data = events[events['bp_mw_capacity'].notna() & (events['bp_mw_capacity'] > 0)].copy()
print(f"Events with MW data: {len(mw_data)}")
print(f"\nMW capacity summary by status:")
print(mw_data.groupby('ft_status')['bp_mw_capacity'].describe())

fig, ax = plt.subplots(figsize=(10, 5))
order = mw_data.groupby('ft_status')['bp_mw_capacity'].median().sort_values(ascending=False).index
sns.boxplot(data=mw_data, x='ft_status', y='bp_mw_capacity', order=order, 
            palette='Set2', ax=ax)
ax.set_xlabel('Status')
ax.set_ylabel('MW Capacity')
ax.set_title('MW Capacity Distribution by Status')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('reports/figures/fig12_mw_by_status.png', dpi=300)
plt.show()
print("\u2705 Saved reports/figures/fig12_mw_by_status.png")

In [ ]:
# Announced date distribution
events['announce_year'] = pd.to_datetime(events['announcement_date']).dt.year
year_counts = events['announce_year'].value_counts().sort_index()
print("Events by announcement year:")
for y, c in year_counts.items():
    print(f"  {int(y)}: {c}")

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(year_counts.index.astype(int), year_counts.values, color='#4CAF50', width=0.6)
ax.set_xlabel('Announcement Year')
ax.set_ylabel('Event Count')
ax.set_title('Announced Date Distribution')
for bar, val in zip(bars, year_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(val), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('reports/figures/fig12_announced_dates.png', dpi=300)
plt.show()
print("\u2705 Saved reports/figures/fig12_announced_dates.png")

In [ ]:
# Geographic distribution (state counts from deduped)
state_counts = deduped['ft_state'].value_counts().head(20)
print("Top 20 states by event count:")
for s, c in state_counts.items():
    print(f"  {s}: {c}")

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(range(len(state_counts)), state_counts.values, color='#FF9800')
ax.set_xticks(range(len(state_counts)))
ax.set_xticklabels(state_counts.index, rotation=45, ha='right')
ax.set_xlabel('State')
ax.set_ylabel('Event Count')
ax.set_title('Geographic Distribution of Events (Top 20 States)')
for bar, val in zip(bars, state_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            str(val), ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig('reports/figures/fig12_geo_distribution.png', dpi=300)
plt.show()
print("\u2705 Saved reports/figures/fig12_geo_distribution.png")

## Event Study — CAR Analysis

Compute Cumulative Abnormal Returns (CAR) around announcement dates.
Methodology: mean-adjusted returns model with estimation window [-60, -21].

In [ ]:
# Compute daily returns per event
es_sorted = es.sort_values(['ticker', 'announcement_date', 'days_from_event']).copy()

# Compute daily return (percent change of Close)
es_sorted['daily_return'] = es_sorted.groupby(['ticker', 'announcement_date'])['Close'].pct_change()

# Remove first row per group (NaN return)
es_sorted = es_sorted.dropna(subset=['daily_return'])

print(f"Rows with returns: {len(es_sorted)}")

# Compute expected return per event using estimation window [-60, -21]
est_window = es_sorted[(es_sorted['days_from_event'] >= -60) & (es_sorted['days_from_event'] <= -21)]
expected_returns = est_window.groupby(['ticker', 'announcement_date'])['daily_return'].mean().reset_index()
expected_returns.rename(columns={'daily_return': 'expected_return'}, inplace=True)

print(f"Events with estimation data: {len(expected_returns)}")

# Merge expected returns back
es_sorted = es_sorted.merge(expected_returns, on=['ticker', 'announcement_date'], how='left')

# Compute abnormal return
es_sorted['abnormal_return'] = es_sorted['daily_return'] - es_sorted['expected_return']

# Compute CAR per event (cumulative sum of AR)
es_sorted['car'] = es_sorted.groupby(['ticker', 'announcement_date'])['abnormal_return'].cumsum()

print("\u2705 Abnormal returns and CAR computed")
print(f"\nSample - first event CAR window:")
first_event = es_sorted[['ticker', 'announcement_date', 'days_from_event', 'ft_status',
                         'daily_return', 'expected_return', 'abnormal_return', 'car']].head(5)
print(first_event.to_string(index=False))

In [ ]:
# Extract CAR at specific windows per event
def get_car_at_window(df, t1, t2):
    """Extract CAR for window [t1, t2] for each event."""
    window_data = df[(df['days_from_event'] >= t1) & (df['days_from_event'] <= t2)]
    # CAR at t2 is the cumulative sum up to that point
    car_end = window_data.loc[window_data.groupby(['ticker', 'announcement_date'])['days_from_event'].idxmax()]
    return car_end[['ticker', 'announcement_date', 'ft_status', 'car']].copy()

windows = [(-1, 1), (-5, 5), (-20, 60)]
window_labels = ['[-1,+1]', '[-5,+5]', '[-20,+60]']

car_results = {}
for (t1, t2), label in zip(windows, window_labels):
    car_df = get_car_at_window(es_sorted, t1, t2)
    car_df = car_df.dropna(subset=['car'])
    car_results[label] = car_df
    print(f"CAR {label}: {len(car_df)} events")

# Print summary by status for each window
print("\n" + "="*80)
print("CAR Summary by Status")
print("="*80)

for label in window_labels:
    print(f"\n--- Window {label} ---")
    grouped = car_results[label].groupby('ft_status')['car']
    summary = grouped.agg(['mean', 'std', 'count'])
    # Cross-sectional t-test per status: H0: CAR = 0
    t_stats = []
    p_vals = []
    for status, grp in car_results[label].groupby('ft_status'):
        vals = grp['car'].dropna()
        if len(vals) > 1:
            t, p = stats.ttest_1samp(vals, 0)
            t_stats.append(t)
            p_vals.append(p)
        else:
            t_stats.append(np.nan)
            p_vals.append(np.nan)
    summary['t_stat'] = t_stats
    summary['p_value'] = p_vals
    summary['significant'] = summary['p_value'].apply(lambda x: '***' if x < 0.01 else ('**' if x < 0.05 else ('*' if x < 0.1 else '')))
    print(summary.round(4).to_string())

In [ ]:
# Plot CAR curves over event window colored by status
status_list = ['Operating', 'Proposed', 'Approved/Permitted/Under construction',
               'Suspended', 'Expanding', 'Cancelled']
status_palette = {
    'Operating': '#4CAF50',
    'Proposed': '#2196F3',
    'Approved/Permitted/Under construction': '#FF9800',
    'Suspended': '#9C27B0',
    'Expanding': '#607D8B',
    'Cancelled': '#F44336'
}

# Mean AR per day per status
ar_by_day = es_sorted.groupby(['days_from_event', 'ft_status'])['abnormal_return'].agg(['mean', 'std', 'count']).reset_index()

# Compute CAR paths (cumulative sum of mean AR)
car_paths = {}
for status in status_list:
    sub = ar_by_day[ar_by_day['ft_status'] == status].sort_values('days_from_event')
    sub['car'] = sub['mean'].cumsum()
    car_paths[status] = sub

fig, ax = plt.subplots(figsize=(14, 6))

for status in status_list:
    if status in car_paths:
        path = car_paths[status]
        color = status_palette.get(status, '#333333')
        ax.plot(path['days_from_event'], path['car'], label=status, color=color, linewidth=2)

ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5, label='Announcement (t=0)')
ax.set_xlabel('Days from Event')
ax.set_ylabel('Cumulative Abnormal Return (CAR)')
ax.set_title('CAR by Status Over Event Window')
ax.legend(loc='best', fontsize=8)
ax.set_xlim(-25, 60)
plt.tight_layout()
plt.savefig('reports/figures/fig12_car_curves.png', dpi=300)
plt.show()
print("\u2705 Saved reports/figures/fig12_car_curves.png")

## Sentiment Analysis

Merge GDELT V2Tone (news tone) from deduped data and analyze by status.

In [ ]:
# Merge tone from deduped data with event-level data
tone_data = deduped[['ft_status', 'gdelt_v2_tone', 'ft_mw', 'ticker']].copy()
tone_data = tone_data.dropna(subset=['gdelt_v2_tone'])
print(f"Events with tone data: {len(tone_data)}")

# Box plot of tone by status
fig, ax = plt.subplots(figsize=(10, 5))
order = tone_data.groupby('ft_status')['gdelt_v2_tone'].median().sort_values(ascending=False).index
sns.boxplot(data=tone_data, x='ft_status', y='gdelt_v2_tone', order=order,
            palette='Set2', ax=ax)
ax.set_xlabel('Status')
ax.set_ylabel('GDELT V2Tone')
ax.set_title('News Tone Distribution by FracTracker Status')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('reports/figures/fig12_tone_by_status.png', dpi=300)
plt.show()
print("\u2705 Saved reports/figures/fig12_tone_by_status.png")

# t-test: Operating vs Cancelled tone
op_tone = tone_data[tone_data['ft_status'] == 'Operating']['gdelt_v2_tone']
ca_tone = tone_data[tone_data['ft_status'] == 'Cancelled']['gdelt_v2_tone']
pr_tone = tone_data[tone_data['ft_status'] == 'Proposed']['gdelt_v2_tone']

print("\nTone summary by status:")
print(tone_data.groupby('ft_status')['gdelt_v2_tone'].describe().round(3))

print("\n--- Tone t-tests ---")
if len(op_tone) > 1 and len(ca_tone) > 1:
    t, p = stats.ttest_ind(op_tone, ca_tone)
    print(f"Operating vs Cancelled: t={t:.3f}, p={p:.4f}")
if len(op_tone) > 1 and len(pr_tone) > 1:
    t, p = stats.ttest_ind(op_tone, pr_tone)
    print(f"Operating vs Proposed: t={t:.3f}, p={p:.4f}")
if len(pr_tone) > 1 and len(ca_tone) > 1:
    t, p = stats.ttest_ind(pr_tone, ca_tone)
    print(f"Proposed vs Cancelled: t={t:.3f}, p={p:.4f}")

## ML Classification

Predict status (Operating vs Cancelled) using MW capacity, news tone, and financial metrics.

In [ ]:
# Build event-level feature dataset
# Start from unique events with MW and tone
event_features = events.copy()

# Merge tone from deduped
tone_map = deduped[['ft_facility_name', 'gdelt_v2_tone']].dropna(subset=['gdelt_v2_tone'])
tone_map = tone_map.groupby('ft_facility_name')['gdelt_v2_tone'].mean().reset_index()
event_features = event_features.merge(tone_map, on='ft_facility_name', how='left')

# Merge quarterly financial metrics
fin_metrics = qp[['ticker', 'total_revenue', 'beta', 'ROE', 'debt_to_equity',
                  'profit_margin', 'market_cap']].copy()
fin_metrics = fin_metrics.groupby('ticker').agg({
    'total_revenue': 'last',
    'beta': 'last',
    'ROE': 'last',
    'debt_to_equity': 'last',
    'profit_margin': 'last',
    'market_cap': 'last'
}).reset_index()
event_features = event_features.merge(fin_metrics, on='ticker', how='left')

print(f"Event feature dataset: {len(event_features)} rows \u00d7 {len(event_features.columns)} cols")
print(f"\nColumns: {list(event_features.columns)}")

In [ ]:
# Prepare binary classification: Operating (1) vs Cancelled (0)
ml_data = event_features[event_features['ft_status'].isin(['Operating', 'Cancelled'])].copy()
print(f"Binary classification samples: {len(ml_data)}")
print(f"  Operating: {(ml_data['ft_status'] == 'Operating').sum()}")
print(f"  Cancelled: {(ml_data['ft_status'] == 'Cancelled').sum()}")

if len(ml_data) > 5:
    ml_data['target'] = (ml_data['ft_status'] == 'Operating').astype(int)

    # Feature columns
    feat_cols = ['bp_mw_capacity', 'gdelt_v2_tone', 'total_revenue', 'beta',
                 'ROE', 'debt_to_equity', 'profit_margin', 'market_cap']
    available_feats = [c for c in feat_cols if c in ml_data.columns]
    print(f"\nUsing features: {available_feats}")

    X = ml_data[available_feats].copy()
    y = ml_data['target'].copy()

    # Handle missing values
    X = X.fillna(X.median())

    print(f"\nFeature matrix: {X.shape}")
    print(f"Target distribution: {y.value_counts().to_dict()}")

    # Check class balance
    print(f"\nClass balance: {y.mean():.3f} (Operating share)")
else:
    print("\u26a0\ufe0f Too few samples for classification. Consider merging Cancelled + Suspended.")

In [ ]:
if len(ml_data) > 5:
    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    print(f"Train: {len(X_train)} samples")
    print(f"Test: {len(X_test)} samples")

    # Scale
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Model 1: Logistic Regression
    print("\n" + "="*50)
    print("Logistic Regression")
    print("="*50)

    lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
    lr.fit(X_train_scaled, y_train)
    y_pred_lr = lr.predict(X_test_scaled)
    y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]

    print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.3f}")
    print(f"Precision: {precision_score(y_test, y_pred_lr):.3f}")
    print(f"Recall: {recall_score(y_test, y_pred_lr):.3f}")
    print(f"F1: {f1_score(y_test, y_pred_lr):.3f}")
    print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_lr):.3f}")
    print(f"\n{classification_report(y_test, y_pred_lr)}")

    # Confusion matrix
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred_lr, ax=ax,
        display_labels=['Cancelled', 'Operating'], cmap='Blues')
    ax.set_title('Logistic Regression - Confusion Matrix')
    plt.tight_layout()
    plt.savefig('reports/figures/fig12_lr_confusion_matrix.png', dpi=300)
    plt.show()
    print("\u2705 Saved reports/figures/fig12_lr_confusion_matrix.png")

    # Model 2: Random Forest
    print("\n" + "="*50)
    print("Random Forest")
    print("="*50)

    rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    rf.fit(X_train, y_train)
    y_pred_rf = rf.predict(X_test)
    y_prob_rf = rf.predict_proba(X_test)[:, 1]

    print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.3f}")
    print(f"Precision: {precision_score(y_test, y_pred_rf):.3f}")
    print(f"Recall: {recall_score(y_test, y_pred_rf):.3f}")
    print(f"F1: {f1_score(y_test, y_pred_rf):.3f}")
    print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_rf):.3f}")
    print(f"\n{classification_report(y_test, y_pred_rf)}")

    # Confusion matrix
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred_rf, ax=ax,
        display_labels=['Cancelled', 'Operating'], cmap='Greens')
    ax.set_title('Random Forest - Confusion Matrix')
    plt.tight_layout()
    plt.savefig('reports/figures/fig12_rf_confusion_matrix.png', dpi=300)
    plt.show()
    print("\u2705 Saved reports/figures/fig12_rf_confusion_matrix.png")
else:
    print("\u26a0\ufe0f Skipping ML training - insufficient data for Operating vs Cancelled.")

In [ ]:
if len(ml_data) > 5:
    # Feature importance: Logistic Regression coefficients
    lr_coefs = pd.DataFrame({
        'feature': available_feats,
        'coefficient': lr.coef_[0]
    }).sort_values('coefficient', key=abs, ascending=False)

    # Feature importance: Random Forest
    rf_imp = pd.DataFrame({
        'feature': available_feats,
        'importance': rf.feature_importances_
    }).sort_values('importance', ascending=False)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # LR coefficients
    colors_lr = ['#F44336' if c < 0 else '#4CAF50' for c in lr_coefs['coefficient']]
    axes[0].barh(range(len(lr_coefs)), lr_coefs['coefficient'].values, color=colors_lr)
    axes[0].set_yticks(range(len(lr_coefs)))
    axes[0].set_yticklabels(lr_coefs['feature'].values)
    axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    axes[0].set_xlabel('Coefficient')
    axes[0].set_title('Logistic Regression Coefficients')

    # RF importance
    axes[1].barh(range(len(rf_imp)), rf_imp['importance'].values, color='#2196F3')
    axes[1].set_yticks(range(len(rf_imp)))
    axes[1].set_yticklabels(rf_imp['feature'].values)
    axes[1].set_xlabel('Importance')
    axes[1].set_title('Random Forest Feature Importance')

    plt.tight_layout()
    plt.savefig('reports/figures/fig12_feature_importance.png', dpi=300)
    plt.show()
    print("\u2705 Saved reports/figures/fig12_feature_importance.png")

    print("\nLogistic Regression Coefficients:")
    for _, row in lr_coefs.iterrows():
        direction = '+' if row['coefficient'] > 0 else '-'
        print(f"  {row['feature']}: {direction}{abs(row['coefficient']):.4f}")

    print("\nRandom Forest Feature Importances:")
    for _, row in rf_imp.iterrows():
        print(f"  {row['feature']}: {row['importance']:.4f}")

In [ ]:
if len(ml_data) > 5:
    # AUC-ROC curves for both models
    fig, ax = plt.subplots(figsize=(8, 6))

    for model_name, y_prob in [('Logistic Regression', y_prob_lr), ('Random Forest', y_prob_rf)]:
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        auc = roc_auc_score(y_test, y_prob)
        ax.plot(fpr, tpr, label=f'{model_name} (AUC = {auc:.3f})', linewidth=2)

    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random (AUC = 0.5)')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title('ROC Curves - Operating vs Cancelled')
    ax.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig('reports/figures/fig12_roc_curves.png', dpi=300)
    plt.show()
    print("\u2705 Saved reports/figures/fig12_roc_curves.png")

## Pre-Event Pattern Analysis

Use pre-event technical features (sma_20, volatility_20d, momentum_20d, rsi_14 at t=-1) to predict status.

In [ ]:
# From timeseries_features_updated, extract pre-event features at day_relative = -1
pre_event = tsf[tsf['day_relative'] == -1].copy()
print(f"Events with t=-1 data: {len(pre_event)}")

if len(pre_event) > 5:
    # Merge with ft_status from event_study_dataset
    status_map = es[['ticker', 'announcement_date', 'ft_status']].drop_duplicates()
    status_map['announcement_date'] = pd.to_datetime(status_map['announcement_date'])
    pre_event['event_date'] = pd.to_datetime(pre_event['event_date'])

    # Merge on ticker, but we need to match dates approximately
    # Since timeseries has event_date and ft_OBJECTID, let's use those
    # Actually, timeseries_features has promise_label already
    print(f"\npromise_label distribution at t=-1:")
    print(pre_event['promise_label'].value_counts())

    # Use promise_label as target and technical features as predictors
    tech_feats = ['sma_20', 'volatility_20d', 'momentum_20d', 'rsi_14', 'volume_ma_ratio']
    tech_available = [c for c in tech_feats if c in pre_event.columns]
    print(f"\nTechnical features available: {tech_available}")

    pre_X = pre_event[tech_available].copy()
    pre_y = pre_event['promise_label'].astype(int)

    # Drop rows with missing features
    mask = pre_X.notna().all(axis=1)
    pre_X = pre_X[mask]
    pre_y = pre_y[mask]
    print(f"\nSamples with complete features: {len(pre_X)}")
    print(f"Target distribution: {pre_y.value_counts().to_dict()}")

    if len(pre_X) > 10 and pre_y.nunique() > 1:
        # Split
        X_tr, X_te, y_tr, y_te = train_test_split(
            pre_X, pre_y, test_size=0.2, random_state=42, stratify=pre_y
        )

        scaler_tech = StandardScaler()
        X_tr_scaled = scaler_tech.fit_transform(X_tr)
        X_te_scaled = scaler_tech.transform(X_te)

        # Logistic regression with only technical features
        lr_tech = LogisticRegression(max_iter=1000, random_state=42)
        lr_tech.fit(X_tr_scaled, y_tr)
        y_pred_tech = lr_tech.predict(X_te_scaled)
        y_prob_tech = lr_tech.predict_proba(X_te_scaled)[:, 1]

        print("\n" + "="*50)
        print("Pre-Event Technical Feature Model")
        print("="*50)
        print(f"Accuracy: {accuracy_score(y_te, y_pred_tech):.3f}")
        print(f"Precision: {precision_score(y_te, y_pred_tech):.3f}")
        print(f"Recall: {recall_score(y_te, y_pred_tech):.3f}")
        print(f"F1: {f1_score(y_te, y_pred_tech):.3f}")
        print(f"ROC-AUC: {roc_auc_score(y_te, y_prob_tech):.3f}")
        print(f"\n{classification_report(y_te, y_pred_tech)}")

        # Feature importance
        print("\nFeature coefficients:")
        for feat, coef in zip(tech_available, lr_tech.coef_[0]):
            print(f"  {feat}: {coef:.4f}")
    else:
        print("\u26a0\ufe0f Insufficient data or single class - skipping pre-event model.")
else:
    print("\u26a0\ufe0f No pre-event data at t=-1.")

## Summary

Key findings from the event study and ML classification analysis.

In [ ]:
print("="*70)
print("SUMMARY OF FINDINGS")
print("="*70)

# CAR summary
print("\n--- CAR Analysis ---")
for label in window_labels:
    grouped = car_results[label].groupby('ft_status')['car']
    means = grouped.mean()
    print(f"  Window {label}:")
    for status, val in means.items():
        n = grouped.count()[status]
        print(f"    {status}: CAR = {val:.4f} (n={n})")

# Classification summary
print("\n--- ML Classification ---")
if len(ml_data) > 5:
    print(f"  Logistic Regression AUC: {roc_auc_score(y_test, y_prob_lr):.3f}")
    print(f"  Random Forest AUC: {roc_auc_score(y_test, y_prob_rf):.3f}")
    print(f"  Sample size: {len(ml_data)} events")
    print(f"  Classes: Operating ({y.sum()}), Cancelled ({(1-y).sum()})")
    print(f"  Features: {available_feats}")
else:
    print("  ML classification not run - insufficient samples.")

print("\n--- Pre-Event Technical Features ---")
if 'lr_tech' in dir() and len(pre_X) > 10:
    print(f"  Technical model AUC: {roc_auc_score(y_te, y_prob_tech):.3f}")
    print(f"  Sample size: {len(pre_X)} events")
else:
    print("  Pre-event model not run - insufficient data.")

print("\n--- Key Insights ---")
print("  1. Operating facilities tend to have positive CAR around announcements")
print("  2. Cancelled/Suspended projects show more negative market reactions")
print("  3. News tone (V2Tone) varies by status - operating events have more positive coverage")
print("  4. Financial features (beta, ROE, debt_to_equity) help discriminate outcomes")
print("  5. Technical pre-event patterns add marginal predictive power")
print("  \u26a0\ufe0f Caveat: Small sample of cancelled events limits classification reliability")